# Sweep Explorer

Choose a MissLearn model family and sweep the missing rate from 10% to
50%, re-injecting an independent MAR pattern at each rate and repeating
the full six-arm, five-fold protocol.

As in the benchmark explorer the model class is held fixed and only the
missing-data treatment varies, so the curves compare strategies rather
than learners. A complete-data run (0% missing) is included as the
reference the degradation plots measure against.

Tick **save this run to disk** beside the family selector to keep every
figure and table this run produces, under `explorer_output/`. Left
unticked nothing is written, which is the default.

---
## 0. Setup

In [3]:
import sys, os, pathlib, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

_BENCH = pathlib.Path.cwd()
sys.path.insert(0, str(_BENCH.parent))   # the MissLearn package
sys.path.insert(0, str(_BENCH))          # benchmark_core, family_registry

import benchmark_core as bc
import family_registry as fr

# ---- Readability ------------------------------------------------------
# Large enough to read on a projector or in a GitHub preview, with padding
# so titles never sit on top of the axes.
plt.rcParams.update({
    "font.size": 13,
    "axes.titlesize": 15,
    "axes.labelsize": 13.5,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "figure.titlesize": 16,
    "axes.titlepad": 12,
    "axes.labelpad": 8,
    "figure.constrained_layout.use": True,
    "figure.dpi": 110,
    "savefig.bbox": "tight",
})
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.precision", 4)

# Saving is controlled by the checkbox beside the family dropdown below,
# not by a constant here. explorer_io routes every figure the notebook draws
# through one place that knows whether to write it.
import explorer_io as eio
eio.install()

print("Ready. Available model families:\n")
fr.describe()

Ready. Available model families:

KEY             MODEL CLASS                            SWEEP?
----------------------------------------------------------------------
MissLinear      Linear / logistic regression           yes
MissRidge       Ridge (L2 penalized)                   yes
MissLASSO       LASSO (L1 penalized)                   yes
MissBayes       Generative Gaussian                    yes
MissNeighbors   k-nearest neighbours (expected distance) yes
MissSupport     Support vector (expected kernel)       yes
MissGaussian    Gaussian process (marginalised kernel) yes


---
## 1. Choose a family and the rate grid

In [9]:
# ======================================================================
#  CHOOSE WHAT TO RUN
#  Pick from the dropdown. The dropdown IS the selection: the cells below
#  read it when they run, so change it and simply run on. There is no need
#  to re-run this cell, and re-running it keeps your choice rather than
#  resetting it.
# ======================================================================
#FAMILY_FALLBACK = "MissBayes"   # used only when ipywidgets is unavailable
N_FOLDS  = 5
RNG_SEED = 42
MISS_RATES = [0.10, 0.20, 0.30, 0.40, 0.50]

try:
    import ipywidgets as W
    _HAVE_WIDGETS = True
except ImportError:
    _HAVE_WIDGETS = False

if _HAVE_WIDGETS:
    _opts = [(f"{k}  --  {v['label']}", k)
             for k, v in fr.FAMILIES.items() if v.get('has_sweep')]
    _keys = [k for _, k in _opts]
    # Reuse an existing dropdown so re-running this cell does not discard a
    # selection already made.
    _prev = globals().get("_dd", None)
    if _prev is not None and getattr(_prev, "value", None) in _keys:
        _start = _prev.value
    else:
        _start = FAMILY_FALLBACK if FAMILY_FALLBACK in _keys else _keys[0]
    _dd = W.Dropdown(options=_opts, value=_start, description="Family:",
                     layout=W.Layout(width="640px"),
                     style={"description_width": "90px"})
    display(_dd)
else:
    _dd = None
    print("ipywidgets not installed; edit FAMILY_FALLBACK above instead.")


if _HAVE_WIDGETS:
    # Reused on re-run for the same reason as the dropdown: rebuilding it
    # would silently untick a box the user had ticked.
    _prev_save = globals().get("_save", None)
    _save = W.Checkbox(
        value=bool(getattr(_prev_save, "value", False)),
        description="save this run to disk", indent=False)
    display(_save)
else:
    _save = None
    SAVE = False        # set True to write output when widgets are absent


def saving_enabled():
    """Whether this run should be written to disk.

    The checkbox is the switch when widgets are available; SAVE is the
    fallback for when they are not. Read at the moment the run starts, so
    ticking the box does not require re-running this cell.
    """
    if _HAVE_WIDGETS and _save is not None:
        return bool(_save.value)
    return bool(globals().get("SAVE", False))


def current_family():
    """The family to run.

    Read from the widget every time rather than cached in a global, so a
    change in the dropdown takes effect on the next cell without this one
    needing to run again.
    """
    if _HAVE_WIDGETS and _dd is not None:
        return _dd.value
    return FAMILY_FALLBACK


print("Current selection:", current_family(),
      "->", fr.FAMILIES[current_family()]["label"])


Dropdown(description='Family:', layout=Layout(width='640px'), options=(('MissLinear  --  Linear / logistic reg…

Checkbox(value=True, description='save this run to disk', indent=False)

Current selection: MissLinear -> Linear / logistic regression


---
## 2. Run the sweep

In [12]:
# Read the dropdown now, not whatever a global was last set to.
FAMILY = current_family()
spec = fr.FAMILIES[FAMILY]

# Opens the output directory when the save box is ticked, and
# resets the figure counter so this run does not append to the last.
OUT_DIR = eio.begin(FAMILY, save=saving_enabled())
display(Markdown(f"## {spec['label']}\n\n{spec['blurb']}"))

# The Gaussian process is O(n^3), so it uses the small datasets only.
_small_only = spec.get("small_only", False)
_n_ds = 1 if _small_only else 2
print(f"Model class held fixed; only the missing-data strategy varies.")
print(f"Datasets: {'small only (O(n^3) model)' if _small_only else 'small + medium'}")

Saving this run to C:\Users\Amanda\Favorites\Machine Learning\MissLearn\benchmarks\explorer_output\MissLinear\2026-08-20_094041


## Linear / logistic regression

Exact FIML linear and logistic regression. The reference family: on linear ground truth it should sit at the efficiency bound, so parity with good imputation is the expected result.

Model class held fixed; only the missing-data strategy varies.
Datasets: small + medium


In [14]:
reg_datasets = bc.make_regression_datasets(mar_rate=MISS_RATES[0],
                                           seed=RNG_SEED)[:_n_ds]
clf_datasets = bc.make_classification_datasets(mar_rate=MISS_RATES[0],
                                               seed=RNG_SEED)[:_n_ds]
print("Sweeping:")
for d in reg_datasets + clf_datasets:
    print(f"  {d['name']:<24} n={d['n']:,}  p={d['p']}")


def run_sweep(datasets, sk, fiml, task):
    """Sweep the rate grid, plus a complete-data reference at 0%."""
    agg, zero, raw = {}, {}, {}
    for ds in datasets:
        print(f"  {ds['name']} ... ", end="", flush=True)
        r = bc.run_cv_at_rate(ds, sk, fiml, task,
                              miss_rates=MISS_RATES,
                              n_splits=N_FOLDS, rng_seed=RNG_SEED)
        raw[ds["name"]] = r
        agg[ds["name"]] = {k: bc.aggregate_folds(v)
                           for k, v in r.items()}
        z = bc.run_cv_at_rate(ds, sk, fiml, task, miss_rates=[0.0],
                              n_splits=N_FOLDS, rng_seed=RNG_SEED)
        zero[ds["name"]] = bc.aggregate_folds(z[0.0])
        print("done")
    return agg, zero, raw

Sweeping:
  Regression-Small         n=300  p=5
  Regression-Medium        n=1,500  p=10
  Classification-Small     n=400  p=5
  Classification-Medium    n=1,500  p=10


In [ ]:
print(f"Regression sweep over {[f'{r*100:.0f}%' for r in MISS_RATES]}")
reg_sweep, reg_zero, reg_raw = run_sweep(
    reg_datasets, spec["reg_sklearn"], spec["reg_fiml"], "regression")

print(f"\nClassification sweep over {[f'{r*100:.0f}%' for r in MISS_RATES]}")
clf_sweep, clf_zero, clf_raw = run_sweep(
    clf_datasets, spec["clf_sklearn"], spec["clf_fiml"],
    "classification")

Regression sweep over ['10%', '20%', '30%', '40%', '50%']
  Regression-Small ... done
  Regression-Medium ... 

---
## 3. Metric against missing rate

In [ ]:
for fig in bc.plot_sweep_lines(reg_datasets, reg_sweep,
                               bc.REG_METRIC_META,
                               model_name=spec["reg_name"],
                               task="regression",
                               miss_rates=MISS_RATES,
                               n_splits=N_FOLDS):
    plt.show()

In [ ]:
for fig in bc.plot_sweep_lines(clf_datasets, clf_sweep,
                               bc.CLF_METRIC_META,
                               model_name=spec["clf_name"],
                               task="classification",
                               miss_rates=MISS_RATES,
                               n_splits=N_FOLDS):
    plt.show()

---
## 4. Degradation from complete data

How much each strategy loses relative to the same model fitted on the
complete matrix.

In [ ]:
for fig in bc.plot_sweep_degradation(reg_datasets, reg_sweep, reg_zero,
                                     bc.REG_METRIC_META,
                                     model_name=spec["reg_name"],
                                     task="regression",
                                     miss_rates=MISS_RATES):
    plt.show()
for fig in bc.plot_sweep_degradation(clf_datasets, clf_sweep, clf_zero,
                                     bc.CLF_METRIC_META,
                                     model_name=spec["clf_name"],
                                     task="classification",
                                     miss_rates=MISS_RATES):
    plt.show()

---
## 5. Ranking and gain

In [ ]:
_ = bc.plot_sweep_rank(reg_datasets, reg_sweep, "r2",
                       model_name=spec["reg_name"], task="regression",
                       miss_rates=MISS_RATES)
plt.show()
_ = bc.plot_sweep_rank(clf_datasets, clf_sweep, "auc",
                       model_name=spec["clf_name"],
                       task="classification", miss_rates=MISS_RATES)
plt.show()

In [ ]:
_ = bc.plot_sweep_gain_heatmap(reg_datasets, reg_sweep,
                               bc.REG_METRIC_META,
                               model_name=spec["reg_name"],
                               task="regression",
                               miss_rates=MISS_RATES)
plt.show()
_ = bc.plot_sweep_gain_heatmap(clf_datasets, clf_sweep,
                               bc.CLF_METRIC_META,
                               model_name=spec["clf_name"],
                               task="classification",
                               miss_rates=MISS_RATES)
plt.show()

---
## 6. Crossover

The lowest rate at which the full-information arm first beats each
conventional strategy on each metric.

In [ ]:
print("Regression:")
_x_reg = bc.compute_sweep_crossover(reg_datasets, reg_sweep,
                                    bc.REG_METRIC_META, MISS_RATES)
eio.save_table(_x_reg, "crossover_regression")
display(_x_reg)
print("Classification:")
_x_clf = bc.compute_sweep_crossover(clf_datasets, clf_sweep,
                                    bc.CLF_METRIC_META, MISS_RATES)
eio.save_table(_x_clf, "crossover_classification")
display(_x_clf)

---
## How to read this

* The interesting question is the **shape** of each curve, not its
  height: the panels use different scales and different datasets.
* Listwise deletion falls away fastest, because the complete-case
  subset shrinks roughly geometrically as the rate rises.
* Where FIML and MICE track each other closely, the honest reading is
  parity maintained across the range rather than an advantage. Look at
  the high-rate end for the real difference: imputation pipelines
  become unstable well before marginalisation does.
* Tick the save box beside the selector to keep this run's figures
  and tables under `explorer_output/`.
* To run every family in one go, use the scripts in `scripts/`.

In [ ]:
# Write the run record and report what was kept. Harmless when the
# save box is unticked: it just says nothing was written.
eio.finish()
